# Streaming statistics on payments
**EN** — Compute live KPIs over a payment stream with bounded memory: running mean/deviation, percentiles, distinct customers (HyperLogLog), top merchants (Space-Saving) and tumbling-window summaries.

**ID** — Menghitung KPI secara langsung dari aliran pembayaran dengan memori terbatas: rata-rata/deviasi berjalan, persentil, jumlah pelanggan unik (HyperLogLog), merchant teratas (Space-Saving), dan ringkasan per jendela waktu.

Requires / Membutuhkan: `bigpiped --mode dev`.

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

In [ ]:
// Using a local build instead of nuget.org? Uncomment and point to <repo>/artifacts/nuget (absolute path):
// #i "nuget: C:/src/bigpipe/artifacts/nuget"
#r "nuget: BigPipe.Analytics, 0.1.0"
using BigPipe.Client;
using BigPipe.Client.Admin;
using BigPipe.Analytics;
var bootstrap = Environment.GetEnvironmentVariable("BIGPIPE_BOOTSTRAP") ?? "localhost:9092";
var admin = new BigPipeAdminClient(Environment.GetEnvironmentVariable("BIGPIPE_ADMIN") ?? "http://localhost:9644");
var http = new BigPipeHttpClient(Environment.GetEnvironmentVariable("BIGPIPE_HTTP") ?? "http://localhost:8082");
string Unique(string p) => $"nb-{p}-{Guid.NewGuid().ToString("N")[..6]}";
var cluster = await admin.GetClusterAsync();
Console.WriteLine($"Connected to {cluster.ClusterId} (BigPipe {cluster.Version}) — {cluster.Credit}");

## 1. Produce a realistic payment stream / Kirim aliran pembayaran
Payments in IDR from merchants across Indonesian cities, with event timestamps spread over five minutes.

In [ ]:
var topic = Unique("payments");
await admin.CreateTopicAsync(topic, 6);
var rng = new Random(42);
var merchants = new[] { "Warung Kopi", "Toko Batik", "Bakmi Jogja", "Sate Madura", "Apotek Sehat", "Gadget BDG" };
var cities = new[] { "Jakarta", "Bandung", "Surabaya", "Yogyakarta", "Medan" };
var t0 = DateTimeOffset.UtcNow.AddMinutes(-5);
double NextExponential(Random r) => -Math.Log(1 - r.NextDouble());
double NextLogNormal(Random r, double mu, double sigma) =>
    Math.Exp(mu + sigma * Math.Sqrt(-2 * Math.Log(1 - r.NextDouble())) * Math.Cos(2 * Math.PI * r.NextDouble()));
var records = Enumerable.Range(0, 3000).Select(i => new HttpProduceRecord
{
    Key = $"cust-{rng.Next(900)}",
    Value = new { merchant = merchants[(int)Math.Min(5, NextExponential(rng) * 1.5)], city = cities[rng.Next(cities.Length)], amount = Math.Round(NextLogNormal(rng, 11.5, 0.8)) },
    Timestamp = t0.AddMilliseconds(i * 100).ToUnixTimeMilliseconds(),
}).ToList();
foreach (var chunk in records.Chunk(500)) await http.ProduceAsync(topic, chunk);
Console.WriteLine($"produced {records.Count} payments to {topic}");

## 2. One pass, bounded memory / Satu lintasan, memori terbatas

In [ ]:
var stats = new RunningStats();
var q = new QuantileSketch();
var customers = new HyperLogLog();
var top = new TopK(20);
var n = 0;
await foreach (var r in BigPipeSource.Kafka(bootstrap, topic, from: OffsetReset.Earliest))
{
    var amount = r.Num("amount");
    stats.Add(amount);
    q.Add(amount);
    customers.Add(r.Key!);
    top.Add(r.Str("merchant")!);
    if (++n == records.Count) break;
}
Console.WriteLine($"payments        {stats.Count:N0}");
Console.WriteLine($"total           {stats.Sum:N0} IDR");
Console.WriteLine($"mean / sd       {stats.Mean:N0} / {stats.StdDev:N0} IDR");
Console.WriteLine($"p50 / p95 / p99 {q.Median:N0} / {q.P95:N0} / {q.P99:N0} IDR");
Console.WriteLine($"unique customers ≈ {customers.Estimate():N0}");
top.Top(5)

## 3. One-minute tumbling windows / Jendela satu menit

In [ ]:
var windows = await BigPipeSource.Kafka(bootstrap, topic, from: OffsetReset.Earliest)
    .TakeAsync(records.Count)
    .TumblingWindow(TimeSpan.FromMinutes(1))
    .CollectAsync();
windows.Select(w => WindowSummary.Of(w, "amount")).Select(w => new { start = w.Start.ToString("HH:mm"), w.Count, total = w.Sum.ToString("N0"), p95 = w.P95.ToString("N0") })